In [0]:
%sql
DESCRIBE shopstream.core.bronze_orders

col_name,data_type,comment
order_line_id,string,null
order_id,string,null
customer_id,string,null
product_id,string,null
quantity,int,null
unit_price,double,null
order_ts,timestamp,null
status,string,null
coupon_code,string,null
# Clustering Information,,


In [0]:
%sql
SELECT * 
FROM shopstream.core.bronze_orders
LIMIT 10;

order_line_id,order_id,customer_id,product_id,quantity,unit_price,order_ts,status,coupon_code
L0000001,O000001,C00628,P0123,1,106.28,2026-02-20T07:14:43.000Z,completed,FESTIVE20
L0000002,O000001,C00628,P0096,1,267.14,2026-02-20T07:14:43.000Z,completed,WELCOME15
L0000003,O000002,C00141,P0057,2,22.09,2026-02-07T07:02:26.000Z,completed,null
L0000004,O000002,C00141,P0017,1,53.8,2026-02-07T07:02:26.000Z,completed,FESTIVE20
L0000005,O000002,C00141,P0126,1,107.67,2026-02-07T07:02:26.000Z,completed,FESTIVE20
L0000006,O000002,C00141,P0110,1,27.62,2026-02-07T07:02:26.000Z,completed,null
L0000007,O000003,C00845,P0165,2,63.33,2026-04-24T10:25:39.000Z,cancelled,null
L0000008,O000004,C00463,P0011,1,64.02,2026-04-27T01:56:15.000Z,returned,null
L0000009,O000004,C00463,P0113,1,319.88,2026-04-27T01:56:15.000Z,returned,null
L0000010,O000004,C00463,P0074,1,156.41,2026-04-27T01:56:15.000Z,returned,WELCOME15


Problem 1: the source system logs `statuses` in two casings

In [0]:
%sql
SELECT status, COUNT(*) AS status_count
FROM shopstream.core.bronze_orders
GROUP BY status 
ORDER BY status_count DESC

status,status_count
completed,8480
cancelled,2071
returned,2053
COMPLETED,731
RETURNED,201
CANCELLED,181


Problem 2: 
- the `quantity Column` has negative quantities
- the `order_line_id` has duplicates

In [0]:
%sql
WITH duplicated_order_line_id AS(
    SELECT order_line_id
    FROM shopstream.core.bronze_orders
    GROUP BY order_line_id 
    HAVING COUNT(*) > 1
),
duplicated_count AS(
    SELECT COUNT(*) AS duplicated_line_ids
    FROM duplicated_order_line_id
),
quantities_in_negative AS(
    SELECT COUNT(*) AS neg_quantity
    FROM shopstream.core.bronze_orders
    WHERE quantity < 0
)
SELECT 
    t1.duplicated_line_ids,
    t2.neg_quantity
FROM duplicated_count t1
CROSS JOIN quantities_in_negative t2

duplicated_line_ids,neg_quantity
234,259


In [0]:
%sql
CREATE OR REPLACE TABLE shopstream.core.silver_orders AS 
WITH dedup AS (
    SELECT * 
    FROM shopstream.core.bronze_orders
    QUALIFY ROW_NUMBER() OVER(PARTITION BY order_line_id ORDER BY order_ts DESC) = 1
)
SELECT
    order_line_id,
    order_id,
    customer_id,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(unit_price AS DOUBLE) AS unit_price,
    ROUND(quantity * unit_price, 2) AS line_revenue,
    CAST(order_ts AS TIMESTAMP) AS order_ts,
    LOWER(status) AS status,
    NULLIF(coupon_code, 'N/A') AS coupon_code
FROM dedup 
WHERE quantity > 0;
    

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_rows FROM shopstream.core.silver_orders;

silver_rows
13228


13,228 rows. We started with 13,717, removed the duplicate copies and the negative quantities, and the math checks out. 

In [0]:
%sql
DESCRIBE shopstream.core.bronze_customers 

col_name,data_type,comment
customer_id,string,null
name,string,null
email,string,null
city,string,null
country,string,null
signup_date,date,null
signup_channel,string,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
SELECT * 
FROM shopstream.core.bronze_customers
LIMIT 10

customer_id,name,email,city,country,signup_date,signup_channel
C00001,Divya Sharma,divya.sharma1@example.com,Singapore,SG,2025-05-08,paid_search
C00002,Arjun Kumar,arjun.kumar2@example.com,Berlin,DE,2025-12-18,organic
C00003,Amelia Patel,amelia.patel3@example.com,Mumbai,IN,2025-02-07,paid_search
C00004,Priya Anderson,priya.anderson4@example.com,Hyderabad,IN,2025-12-21,email
C00005,Mason Mehta,mason.mehta5@example.com,San Francisco,US,2025-10-09,organic
C00006,Rohan Davis,rohan.davis6@example.com,Chennai,IN,2025-05-05,paid_search
C00007,Ava Kumar,ava.kumar7@example.com,Bengaluru,IN,2025-07-04,social
C00008,Ethan Lopez,ethan.lopez8@example.com,Pune,IN,2025-01-24,referral
C00009,Divya Miller,divya.miller9@example.com,Bengaluru,IN,2025-09-10,email
C00010,Sophia Martinez,sophia.martinez10@example.com,Hyderabad,IN,2025-12-03,organic


In [0]:
%sql
SELECT country, COUNT(*)
FROM shopstream.core.bronze_customers
GROUP BY 1 

country,COUNT(*)
SG,69
DE,78
IN,392
US,193
AU,71
AE,61
CA,70
GB,66


In [0]:
%sql
CREATE OR REPLACE TABLE shopstream.core.silver_customers AS
SELECT 
    TRIM(customer_id)                           AS customer_id,
    name,
    TRIM(LOWER(email))                          AS email,
    city,
    TRIM(country)                               AS country,
    TO_DATE(TRIM(signup_date), 'yyyy-MM-dd')    AS signup_date,
    TRIM(signup_channel)                        AS signup_channel
FROM shopstream.core.bronze_customers;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS silver_customer_rows FROM shopstream.core.silver_customers

silver_customer_rows
1000


In [0]:
%sql
DESCRIBE shopstream.core.bronze_products 

col_name,data_type,comment
product_id,string,null
product_name,string,null
category,string,null
unit_price,double,null
unit_cost,double,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
SELECT * FROM shopstream.core.bronze_products LIMIT 10

product_id,product_name,category,unit_price,unit_cost
P0001,Wireless Earbuds,electronics,69.01,45.86
P0002,Wireless Earbuds v2,electronics,244.41,127.05
P0003,Wireless Earbuds v3,electronics,41.1,24.58
P0004,Wireless Earbuds v4,electronics,89.69,48.74
P0005,Wireless Earbuds v5,electronics,281.95,211.29
P0006,Mechanical Keyboard,electronics,229.55,105.48
P0007,Mechanical Keyboard v2,electronics,27.92,18.85
P0008,Mechanical Keyboard v3,electronics,40.16,21.28
P0009,Mechanical Keyboard v4,electronics,105.65,64.76
P0010,Mechanical Keyboard v5,electronics,47.69,34.28


In [0]:
%sql
CREATE OR REPLACE TABLE shopstream.core.silver_products AS
SELECT 
    TRIM(product_id) AS product_id,
    product_name,
    category,
    unit_price,
    unit_cost,
    ROUND(unit_price - unit_cost, 2) AS margin
FROM shopstream.core.bronze_products;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) FROM shopstream.core.silver_products;

COUNT(*)
197


In [0]:
%sql
DESCRIBE HISTORY shopstream.core.silver_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-01T05:40:00.000Z,77386382207343,sohomdas.work@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""38dec8c8-5b57-4580-8317-048ba81b8c0b"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-84e1599a-3a2b-4151-8b7e-e95b3923efca"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5fa4cbb1-d3f3-44e6-8c36-8bd05128dd1d""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(4147485632324302),875276cc-ba98-4816-8004-ecfb295d5b9e,1101-051755-bom0pamn-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 13228, numOutputBytes -> 180986)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13


- Business rule: cancelled orders don't belong in silver.

- Delta DML makes this a one-liner, and time travel keeps the audit trail.

In [0]:
%sql
DELETE FROM shopstream.core.silver_orders WHERE status = 'cancelled';

SELECT 
    (SELECT COUNT(*) FROM shopstream.core.silver_orders) AS rows_after_cancellation,
    (SELECT COUNT(*) FROM shopstream.core.silver_orders VERSION AS OF 0) AS rows_before_cancellation

rows_after_cancellation,rows_before_cancellation
11061,13228
